# Load SEC filings for your action

Choose an action and dates, download the filings, and inspect the text in a pandas table. Run the cells from top to bottom. This notebook does not call an LLM.

In [2]:
from pathlib import Path
import sys
import pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'notebooks' / 'evaluation' / 'load_action_filings.py').exists())
sys.path.insert(0, str(ROOT / 'notebooks' / 'evaluation'))
from load_action_filings import ACTIONS, load_action
pd.DataFrame([{'action': action, **spec} for action, spec in ACTIONS.items()])

,action,primary,supporting,conditional,query
0,tender_offer,"[SC TO-T, SC TO-T/A, SC TO-I, SC TO-I/A]","[SC 14D9, SC 14D9/A]",[],"""tender offer"" OR ""offer to purchase"""
1,exchange_offer,"[SC TO-T, SC TO-T/A, SC TO-I, SC TO-I/A, S-4, ...","[SC 14D9, SC 14D9/A, SC TO-C, 425, 8-K, 6-K]","[SC 13E3, SC 13E3/A]","""exchange offer"" OR ""offer to exchange"""
2,rights_issue,"[424B2, 424B3, 424B5, S-3, S-3ASR]","[8-K, 424B4]",[],"""rights offering"" OR ""subscription rights"" OR ..."
3,merger,"[DEFM14A, PREM14A, S-4, S-4/A, SC 13E3, SC 13E...","[8-K, 425]",[],"""merger"" OR ""business combination"""
4,conversion,"[8-K, 424B2, 424B3, S-3, S-3ASR]","[10-K, 10-Q]",[],"""conversion rate"" OR ""conversion price"" OR ""ma..."


## 1. Choose what to download

Edit the settings below. Dates are filing dates; the count is a maximum number of filings, not events.

Before running, set `SEC_USER_AGENT` to your name/project and contact email in the environment running Jupyter. No SEC API key or Chrome is needed.

In [3]:
ACTION = 'exchange_offer'  # tender_offer, exchange_offer, rights_issue, merger, conversion
START = '2023-07-01'
END = '2023-08-31'
N_FILINGS = 1
OUTPUT = ROOT / 'notebooks' / 'evaluation' / 'data' / f'{ACTION}_{START}_{END}'
# Optional: use a saved filing list instead of searching SEC.
METADATA_PATH = None  # ROOT / 'data/processed/documents.parquet'

## 2. Download the files

This cell contacts SEC and reuses files already saved. The forms and keywords come from `corporate_actions.json`.

The default search limit is 100 results per base form (including amendments). Check `search_counts.csv` for limits reached. The sample varies by year/company but may not include every form.

In [5]:
# set env variable for SEC_USER_AGENT="Name email@gmail.com"
filings_df = load_action(
    action=ACTION, start=START, end=END, n=N_FILINGS,
    include_supporting=True, include_conditional=False,
    max_hits_per_form=100, metadata_path=METADATA_PATH, output_dir=OUTPUT,
)
filings_df.drop(columns='text')

ValueError: Set SEC_USER_AGENT to your project/name and contact email

## 3. Check the text

Each row combines a filing and its attached exhibits. The preview below shows only the first 6,000 characters; the saved text is complete for documents successfully processed.

- `ok`: selected documents were processed.
- `partial`: some selected documents could not be read as text.
- `failed`: no document text was produced.

Check that the filing matches your action. Compare complex tables with the original. PDF/image attachments are saved but not converted to text. Documents referenced in other filings are not downloaded automatically, even when status is `ok`.

In [ ]:
if not filings_df.empty:
    print(filings_df[['accession_number', 'status']].to_string(index=False))
    print(filings_df.iloc[0]['text'][:6000])
manifest_path = OUTPUT / 'documents.csv'
if manifest_path.exists():
    display(pd.read_csv(manifest_path)[['accession_number', 'filename', 'document_type', 'role', 'status', 'error']])

## 4. Use the text with your LLM

Use a reviewed row’s `text` as the filing input in `Evaluation_LLM.ipynb`. We are using one LLM for now. The example below stays commented out so it does not call StackAI.

If the text is too long for the model, split it into documented sections first; do not silently cut it off. Downloads and prepared text are saved in the `OUTPUT` folder.

In [ ]:
# After checking the text, use a row in your LLM notebook:
# row = filings_df.iloc[0]
# filing_text = row['text']
# output = query({'in-0': filing_text, 'in-1': action_prompt})
# Attach row['accession_number'] and row['filing_date'] to parsed results.

# Reload saved results without downloading again:
# filings_df = pd.read_json(OUTPUT / 'filings.jsonl', lines=True, convert_dates=False)